In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
from StocProcess.RBM import MakeRBMTransProbFunc
from QAE.LowDepthQAE import LowDepthQAE

In [2]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
t = 0.6
mu = 0.5
sigma = 1.0
n_terms = 5

# QAE setting
nShot = 12
epsilon = 0.0025
nRep = 100

In [3]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(t, t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [4]:
Ns = (2 ** np.linspace(3, 7, 9)).astype(int)
print(Ns)

[  8  11  16  22  32  45  64  90 128]


In [5]:
retDf = pd.DataFrame(columns=['N', 'pTrue', 'beta', 'pEst', 'absErr', 'totalQueryNum', 'maxDepth'])

for _ in range(nRep):
    for iN in range(len(Ns)):
        N = Ns[iN]
        beta = np.log(N**0.5) / np.log(1 / epsilon)
        qaeRes = LowDepthQAE(pTrue, epsilon, nShot, beta)
        pEst = qaeRes.aEst
        totalQueryNum = qaeRes.TotalQueryNum * N * (N+1) / 2
        maxDepth = qaeRes.MaxDepth * N
        retDf.loc[len(retDf)] = [N, pTrue, beta, pEst, abs(pEst - pTrue), totalQueryNum, maxDepth]

c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


In [6]:
retDf

,N,pTrue,beta,pEst,absErr,totalQueryNum,maxDepth
0,8.0,0.649605,0.173534,0.649765,0.00016,516240.0,3000.0
1,11.0,0.649605,0.200109,0.649565,0.00004,1021680.0,3157.0
2,16.0,0.649605,0.231378,0.649465,0.00014,2134656.0,3216.0
3,22.0,0.649605,0.257954,0.649165,0.00044,5486052.0,3982.0
4,32.0,0.649605,0.289223,0.649965,0.00036,13330944.0,4512.0
...,...,...,...,...,...,...,...
895,32.0,0.649605,0.289223,0.649765,0.00016,13330944.0,4512.0
896,45.0,0.649605,0.317674,0.649665,0.00006,34428240.0,5535.0
897,64.0,0.649605,0.347067,0.649565,0.00004,83566080.0,6464.0
898,90.0,0.649605,0.375518,0.650065,0.00046,210073500.0,7650.0


In [7]:
retDf.groupby('N')['absErr'].mean()

N
8.0      0.001190
11.0     0.000219
16.0     0.000269
22.0     0.000285
32.0     0.000275
45.0     0.000232
64.0     0.000243
90.0     0.000216
128.0    0.000248
Name: absErr, dtype: float64

In [8]:
retDf.to_csv('RBM_LowDepthQAE.csv', index=False)